## Local Inference on GPU
Model page: https://huggingface.co/ameer4wisam/gemma-iraqi-finetune

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/ameer4wisam/gemma-iraqi-finetune)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [ ]:
!pip install --upgrade torchao

from peft import PeftModel
from transformers import AutoModelForCausalLM
from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

base_model = AutoModelForCausalLM.from_pretrained("google/gemma-4-12B-it")
model = PeftModel.from_pretrained(base_model, "ameer4wisam/gemma-iraqi-finetune")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 75.0 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

adapter_model.safetensors:   0%|          | 0.00/140M [00:00<?, ?B/s]

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base_model_id = "google/gemma-4-12B-it"
adapter_model_id = "ameer4wisam/gemma-iraqi-finetune"

# تحميل النموذج الأساسي
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.bfloat16,
    device_map={"": 0}   # GPU رقم 0
)

# تحميل LoRA
model = PeftModel.from_pretrained(
    base_model,
    adapter_model_id
)

# دمج LoRA مع النموذج الأساسي
merged_model = model.merge_and_unload()

# حفظ النموذج المدمج
merged_model.save_pretrained("./gemma-iraqi-merged")
AutoTokenizer.from_pretrained(base_model_id).save_pretrained("./gemma-iraqi-merged")

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./gemma-iraqi-merged/tokenizer_config.json',
 './gemma-iraqi-merged/chat_template.jinja',
 './gemma-iraqi-merged/tokenizer.json')

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

model = AutoModelForCausalLM.from_pretrained(
    "./gemma-iraqi-merged",
    torch_dtype=torch.bfloat16,
    device_map={"": 0}
)

tokenizer = AutoTokenizer.from_pretrained("./gemma-iraqi-merged")

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

In [ ]:
import os
# تعطيل Xet التجريبي عبر متغيرات البيئة
os.environ["HF_HUB_USE_XET"] = "0"

# إيقاف Xet برمجياً وبشكل إجباري لتجنب أخطاء الرفع
import huggingface_hub._commit_api
huggingface_hub._commit_api.is_xet_available = lambda: False

from huggingface_hub import login, HfApi
from google.colab import userdata

new_token = userdata.get('ameerwisam2005')
login(token=new_token)

# التحقق من هوية الحساب المرتبط بالرمز
api = HfApi()
user_info = api.whoami(token=new_token)
print(f"✅ تم تسجيل الدخول بحساب: {user_info.get('name', 'Unknown')}")

hf_username = "ameer4wisam"
# استخدام اسم مستودع جديد
repo_id = f"{hf_username}/gemma-iraqi-finetune-v2"

if user_info.get('name') != hf_username:
    print(f"\n❌ خطأ: الرمز يتبع لحساب '{user_info.get('name')}' بينما نحاول الرفع إلى '{hf_username}'.")
    print("يرجى التأكد من إنشاء رمز من نوع Legacy وبصلاحية Write من حساب ameer4wisam.")
else:
    print(f"\nجاري إنشاء المستودع الجديد {repo_id}...")
    # إنشاء المستودع الجديد
    api.create_repo(repo_id=repo_id, exist_ok=True, token=new_token)

    print("جاري رفع النموذج بالوضع التقليدي (بدون Xet)...")
    merged_model.push_to_hub(repo_id, token=new_token)
    tokenizer.push_to_hub(repo_id, token=new_token)
    print("🎉 تم الرفع بنجاح!")

✅ تم تسجيل الدخول بحساب: ameer4wisam

جاري إنشاء المستودع الجديد ameer4wisam/gemma-iraqi-finetune-v2...
جاري رفع النموذج بالوضع التقليدي (بدون Xet)...


README.md:   0%|          | 0.00/8.82k [00:00<?, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
from huggingface_hub import HfApi

# اسم المستودع الخاص بك
repo_id = "ameer4wisam/gemma-iraqi-finetune-v2"

# محتوى بطاقة النموذج (Model Card)
model_card_content = """
---
library_name: transformers
pipeline_tag: text-generation
base_model: google/gemma-4-12B-it
language:
  - ar
  - en
license: gemma
tags:
  - gemma
  - text-generation
  - arabic
  - iraqi
  - merged
---
# 📘 التوثيق الرسمي — نموذج Gemma للهجة العراقية (المدمج v2)

> **المستودع:** [`ameer4wisam/gemma-iraqi-finetune-v2`](https://huggingface.co/ameer4wisam/gemma-iraqi-finetune-v2) (مدمج، جاهز مباشرة)
> **نسخة الـ Adapter المنفصل:** [`ameer4wisam/gemma-iraqi-finetune`](https://huggingface.co/ameer4wisam/gemma-iraqi-finetune) (للمطورين)

---

## 1. نظرة عامة

نموذج محادثة باللهجة العراقية مبني على `google/gemma-4-12B-it` بتقنية LoRA ثم دُمج بالأوزان الأساسية (bf16). متخصص بمحادثات **البيع والشراء والخدمات والحياة اليومية**، ويلتزم بأسلوب الرد القصير المباشر للبائع العراقي.

| البند | القيمة |
|---|
| النموذج الأساسي | `google/gemma-4-12B-it` |
| التقنية | LoRA (r=16, alpha=32, dropout=0.05) ثم دمج bf16 |
| الطبقات المستهدفة | q/k/v/o_proj + gate/up/down_proj (نموذج اللغة فقط، regex يستثني أبراج الرؤية/الصوت) |
| بيانات التدريب | 163,429 محادثة تدريب / 10,330 تقييم (iraqi_train_v4) |
| الحقب | **1 epoch** (20,429 خطوة) |
| معدل التعلم | **5e-5** |
| حجم الدفعة | 8 (A100 40GB) |
| النتائج | Validation Loss: **0.215** — Token Accuracy: **93.8%** |

---

## 2. ⚙️ وصفة الاستدلال الإلزامية

هذه الإعدادات **مثبتة بالاختبارات** — أي انحراف عنها يُنتج مخرجات مشوهة (سلطة كلمات) رغم سلامة النموذج:

| الإعداد | القيمة | السبب |
|---|
| `eos_token_id` | يُكتشف تلقائياً من القالب (id=106 + eos) | كتابة اسم التوكن يدوياً ترجع UNK بصمت والتوليد لا يتوقف |
| `attn_implementation` | `"eager"` | مطلوب لمعمارية Gemma 4 |
| `max_new_tokens` | 64 | أجوبة بيانات التدريب قصيرة (10–30 توكن)؛ الطول الزائد = هذيان |
| الوضع الافتراضي | `do_sample=False` (حتمي) | **إلزامي لأي رد فيه أرقام/أسعار/حقائق** |
| الوضع الإبداعي | temperature=0.3, top_p=0.8, top_k=20 | للتحيات والدردشة فقط |
| `repetition_penalty` | ❌ **ممنوع نهائياً** | يعاقب مفردات اللهجة المتكررة ويدفع لكلمات مخترعة |
| temperature > 0.3 | ❌ ممنوع | التوزيع متخصص وضيق؛ العشوائية الواسعة تكسره |

### مثال استدلال كامل (انسخه كما هو):

```python
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "ameer4wisam/gemma-iraqi-finetune-v2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16,
    device_map="auto", attn_implementation="eager",
).eval()

# اكتشاف توكن التوقف تلقائياً (لا تكتبه يدوياً)
_probe = tokenizer.apply_chat_template(
    [{"role": "user", "content": "هلو"},
     {"role": "assistant", "content": "هلا بيك"}],
    add_generation_prompt=False)
special = set(tokenizer.all_special_ids)
stop_ids = list({int(t) for t in _probe[-3:] if int(t) in special}
                | {tokenizer.eos_token_id})

def chat(messages, deterministic=True):
    enc = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True,
        return_tensors="pt", return_dict=True)
    cfg = (dict(do_sample=False) if deterministic
           else dict(do_sample=True, temperature=0.3, top_p=0.8, top_k=20))
    out = model.generate(
        input_ids=enc["input_ids"].to(model.device),
        attention_mask=enc["attention_mask"].to(model.device),
        eos_token_id=stop_ids, max_new_tokens=64, **cfg)
    return tokenizer.decode(
        out[0][enc["input_ids"].shape[1]:],
        skip_special_tokens=True).strip()
```

---

## 3. 🧾 System Prompt الرسمي (بائع + كتالوج)

النموذج لم يُدرَّب على system prompts، لكن قدرة Gemma الأساسية تجعله يستجيب لها بشكل ممتاز (مثبت بالاختبار). **القاعدة المعمارية:** التعليمات السلوكية بالبرومبت، والحقائق (أسعار/منتجات) تُحقن ككتالوج صريح، والحسابات (خصومات/مجاميع) تُحسب بالكود مسبقاً وتُذكر جاهزة.

```text
أنت بائع عراقي محترف بمحل أجهزة كهربائية.

المنتجات المتوفرة حالياً (التزم بيها حرفياً):
- [يُبنى ديناميكياً من قاعدة البيانات]
- خصم شراء قطعتين: 5% (المجموع بعد الخصم: [محسوب مسبقاً])

قواعد صارمة:
1. جاوب باللهجة العراقية الأصيلة فقط: شنو، شلون، هسه، اكو، ماكو، زين، خوش، شكد.
2. جاوب قصير ومباشر، جملة أو جملتين مثل البائع الحقيقي.
3. الأسعار والأرقام من القائمة أعلاه فقط - لا تخترع أي رقم.
4. بِع فقط المنتجات المذكورة بالقائمة. إذا طلب الزبون منتج غير موجود،
   گله بصراحة: "والله هذا ماكو عدنا هسه" واعرض البديل إذا مناسب.
5. تذكر تفاصيل الزبون (اسمه، شنو يريد) واستعملها بردودك.
6. إذا ما تعرف معلومة، گول "أتأكدلك" ولا تخترع جواب.
```

---

## 4. 📊 نتائج التحقق الموثقة

| الاختبار | النتيجة |
|---|
| محادثة مبيعات 12 دور (حتمي + كتالوج) | ✅ أسعار من الكتالوج، ثابتة عبر الأدوار، لهجة سليمة |
| سياق طويل حتى 785 توكن (~4× أطول مثال تدريب) | ✅ لا تدهور — قدرة السياق من Gemma الأساسي |
| تذكر اسم الزبون وطلبه بعد 5 أدوار | ✅ مع system prompt (بدونه يتذكر الطلب فقط) |
| أسئلة متابعة بالسياق ("وتعطي خصم للاثنين؟") | ✅ يفهم المرجعية من المحادثة |
| التحيات والمجاملات | ✅ طبيعية (أمثلة v4 المضافة) |

## 5. ⚠️ القيود المعروفة

1. **الأرقام بدون كتالوج غير موثوقة** — النموذج يولّد أسعاراً من توزيع التدريب. الحل: حقن الكتالوج بالسياق دائماً (إلزامي للإنتاج).
2. **الحساب ضعيف** — الخصومات والمجاميع تُحسب بالكود وتُحقن جاهزة، لا يُعتمد على النموذج.
3. **العشوائية تفسد الحقائق** — حتى 0.3 قد يُزحزح رقماً عن الكتالوج؛ الردود الحقائقية حتمية إجبارياً.
4. **قد يبيع بديلاً بدل الرفض** — بدون القاعدة 4 بالبرومبت، طلبُ منتجٍ غير متوفر قد يُقابَل بعرض منتج آخر وكأنه المطلوب.
5. **القدرة العامة محدودة** — متخصص بالبيع/الخدمات؛ خارج هذا النطاق الجودة تتفاوت.
6. **لم يُدرَّب على system prompts** — الاستجابة لها من Gemma الأساسي؛ قد تضعف مع برومبتات معقدة جداً.

## 6. 🔁 سجل الدروس التقنية (لإعادة التدريب مستقبلاً)

- **لا تضع `use_cache=False` أبداً** مع Gemma 4 12B (يفسد مشاركة KV بين الطبقات — transformers issue #45242). كذلك `gradient_checkpointing=False` إلزامي لنفس السبب.
- `target_modules` بقائمة صريحة عبر regex يستثني أبراج الرؤية/الصوت — وليس `"all-linear"`.
- الدمج فوق **bf16 فقط** — الدمج فوق أوزان مكممة (4/8-bit) يراكم أخطاء ويخرب النموذج.
- بعد أي دمج: اختبار مقارنة حتمي قبل/بعد بنفس الأسئلة (Gemma 4 12B حساسة معمارياً).
- لبيانات v5 المقترحة: أمثلة رفض، أمثلة التزام بكتالوج محقون، أمثلة تذكّر تفاصيل الزبون، محادثات أطول (400–800 توكن).
"""

# حفظ المحتوى في ملف README.md محلياً
with open("README.md", "w", encoding="utf-8") as f:
    f.write(model_card_content)

print("جاري رفع بطاقة النموذج (Model Card) المحدثة إلى Hugging Face...")

# رفع الملف إلى Hugging Face
api = HfApi()
api.upload_file(
    path_or_fileobj="README.md",
    path_in_repo="README.md",
    repo_id=repo_id,
    repo_type="model",
)

print("✅ تم تحديث ورفع بطاقة النموذج بنجاح!")

### تجربة النموذج (Inference)

هذا الكود يقوم بتحميل النموذج المدمج الذي قمنا برفعه للتو ويستخدمه للإجابة على الأسئلة باللهجة العراقية بناءً على التوجيه (System Prompt) المخصص.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# تحديد اسم النموذج النهائي المرفوع على Hugging Face
model_id = "ameer4wisam/gemma-iraqi-finetune-v2"

print("جاري تحميل النموذج ومجزئ الرموز...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

# إعداد خط الأنابيب (Pipeline) لتوليد النصوص
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

# إعداد التوجيه الخاص باللهجة العراقية
IRAQI_SYSTEM_PROMPT = """
أنت مساعد ذكاء اصطناعي عراقي.

القواعد:
- تكلم دائماً باللهجة العراقية الطبيعية.
- استخدم مفردات عراقية مثل: شنو، شلون، هسه، كلش، مو، أني، إنت، خوش، بعد، عبالك.
- لا تستخدم العربية الفصحى إلا إذا طلب المستخدم ذلك.
- إذا كتب المستخدم بالفصحى، أجب بالعراقي أيضاً ما لم يطلب غير ذلك.
- حافظ على الأسلوب الودود والمحترم والمساعد.
- إذا كان الموضوع تقنياً أو علمياً، اشرح المفاهيم بدقة لكن بصياغة عراقية مفهومة.
"""

# رسالة المستخدم
messages = [
    {"role": "system", "content": IRAQI_SYSTEM_PROMPT},
    {"role": "user", "content": "شلونك شخبارك؟ شنو رأيك بالذكاء الاصطناعي؟"}
]

print("\nجاري توليد الإجابة...")
outputs = pipe(messages, max_new_tokens=256)

print("\n--- إجابة النموذج ---")
print(outputs[0]["generated_text"][-1]["content"])

# 📘 التوثيق الرسمي — نموذج Gemma للهجة العراقية (المدمج v2)

> **المستودع:** [`ameer4wisam/gemma-iraqi-finetune-v2`](https://huggingface.co/ameer4wisam/gemma-iraqi-finetune-v2) (مدمج، جاهز مباشرة)
> **نسخة الـ Adapter المنفصل:** [`ameer4wisam/gemma-iraqi-finetune`](https://huggingface.co/ameer4wisam/gemma-iraqi-finetune) (للمطورين)

---

## 1. نظرة عامة

نموذج محادثة باللهجة العراقية مبني على `google/gemma-4-12B-it` بتقنية LoRA ثم دُمج بالأوزان الأساسية (bf16). متخصص بمحادثات **البيع والشراء والخدمات والحياة اليومية**، ويلتزم بأسلوب الرد القصير المباشر للبائع العراقي.

| البند | القيمة |
|---|---|
| النموذج الأساسي | `google/gemma-4-12B-it` |
| التقنية | LoRA (r=16, alpha=32, dropout=0.05) ثم دمج bf16 |
| الطبقات المستهدفة | q/k/v/o_proj + gate/up/down_proj (نموذج اللغة فقط، regex يستثني أبراج الرؤية/الصوت) |
| بيانات التدريب | 163,429 محادثة تدريب / 10,330 تقييم (iraqi_train_v4) |
| الحقب | **1 epoch** (20,429 خطوة) |
| معدل التعلم | **5e-5** |
| حجم الدفعة | 8 (A100 40GB) |
| النتائج | Validation Loss: **0.215** — Token Accuracy: **93.8%** |

---

## 2. ⚙️ وصفة الاستدلال الإلزامية

هذه الإعدادات **مثبتة بالاختبارات** — أي انحراف عنها يُنتج مخرجات مشوهة (سلطة كلمات) رغم سلامة النموذج:

| الإعداد | القيمة | السبب |
|---|---|---|
| `eos_token_id` | يُكتشف تلقائياً من القالب (id=106 + eos) | كتابة اسم التوكن يدوياً ترجع UNK بصمت والتوليد لا يتوقف |
| `attn_implementation` | `"eager"` | مطلوب لمعمارية Gemma 4 |
| `max_new_tokens` | 64 | أجوبة بيانات التدريب قصيرة (10–30 توكن)؛ الطول الزائد = هذيان |
| الوضع الافتراضي | `do_sample=False` (حتمي) | **إلزامي لأي رد فيه أرقام/أسعار/حقائق** |
| الوضع الإبداعي | temperature=0.3, top_p=0.8, top_k=20 | للتحيات والدردشة فقط |
| `repetition_penalty` | ❌ **ممنوع نهائياً** | يعاقب مفردات اللهجة المتكررة ويدفع لكلمات مخترعة |
| temperature > 0.3 | ❌ ممنوع | التوزيع متخصص وضيق؛ العشوائية الواسعة تكسره |

### مثال استدلال كامل (انسخه كما هو):

```python
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "ameer4wisam/gemma-iraqi-finetune-v2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16,
    device_map="auto", attn_implementation="eager",
).eval()

# اكتشاف توكن التوقف تلقائياً (لا تكتبه يدوياً)
_probe = tokenizer.apply_chat_template(
    [{"role": "user", "content": "هلو"},
     {"role": "assistant", "content": "هلا بيك"}],
    add_generation_prompt=False)
special = set(tokenizer.all_special_ids)
stop_ids = list({int(t) for t in _probe[-3:] if int(t) in special}
                | {tokenizer.eos_token_id})

def chat(messages, deterministic=True):
    enc = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True,
        return_tensors="pt", return_dict=True)
    cfg = (dict(do_sample=False) if deterministic
           else dict(do_sample=True, temperature=0.3, top_p=0.8, top_k=20))
    out = model.generate(
        input_ids=enc["input_ids"].to(model.device),
        attention_mask=enc["attention_mask"].to(model.device),
        eos_token_id=stop_ids, max_new_tokens=64, **cfg)
    return tokenizer.decode(
        out[0][enc["input_ids"].shape[1]:],
        skip_special_tokens=True).strip()
```

---

## 3. 🧾 System Prompt الرسمي (بائع + كتالوج)

النموذج لم يُدرَّب على system prompts، لكن قدرة Gemma الأساسية تجعله يستجيب لها بشكل ممتاز (مثبت بالاختبار). **القاعدة المعمارية:** التعليمات السلوكية بالبرومبت، والحقائق (أسعار/منتجات) تُحقن ككتالوج صريح، والحسابات (خصومات/مجاميع) تُحسب بالكود مسبقاً وتُذكر جاهزة.

```text
أنت بائع عراقي محترف بمحل أجهزة كهربائية.

المنتجات المتوفرة حالياً (التزم بيها حرفياً):
- [يُبنى ديناميكياً من قاعدة البيانات]
- خصم شراء قطعتين: 5% (المجموع بعد الخصم: [محسوب مسبقاً])

قواعد صارمة:
1. جاوب باللهجة العراقية الأصيلة فقط: شنو، شلون، هسه، اكو، ماكو، زين، خوش، شكد.
2. جاوب قصير ومباشر، جملة أو جملتين مثل البائع الحقيقي.
3. الأسعار والأرقام من القائمة أعلاه فقط - لا تخترع أي رقم.
4. بِع فقط المنتجات المذكورة بالقائمة. إذا طلب الزبون منتج غير موجود،
   گله بصراحة: "والله هذا ماكو عدنا هسه" واعرض البديل إذا مناسب.
5. تذكر تفاصيل الزبون (اسمه، شنو يريد) واستعملها بردودك.
6. إذا ما تعرف معلومة، گول "أتأكدلك" ولا تخترع جواب.
```

---

## 4. 📊 نتائج التحقق الموثقة

| الاختبار | النتيجة |
|---|---|
| محادثة مبيعات 12 دور (حتمي + كتالوج) | ✅ أسعار من الكتالوج، ثابتة عبر الأدوار، لهجة سليمة |
| سياق طويل حتى 785 توكن (~4× أطول مثال تدريب) | ✅ لا تدهور — قدرة السياق من Gemma الأساسي |
| تذكر اسم الزبون وطلبه بعد 5 أدوار | ✅ مع system prompt (بدونه يتذكر الطلب فقط) |
| أسئلة متابعة بالسياق ("وتعطي خصم للاثنين؟") | ✅ يفهم المرجعية من المحادثة |
| التحيات والمجاملات | ✅ طبيعية (أمثلة v4 المضافة) |

## 5. ⚠️ القيود المعروفة

1. **الأرقام بدون كتالوج غير موثوقة** — النموذج يولّد أسعاراً من توزيع التدريب. الحل: حقن الكتالوج بالسياق دائماً (إلزامي للإنتاج).
2. **الحساب ضعيف** — الخصومات والمجاميع تُحسب بالكود وتُحقن جاهزة، لا يُعتمد على النموذج.
3. **العشوائية تفسد الحقائق** — حتى 0.3 قد يُزحزح رقماً عن الكتالوج؛ الردود الحقائقية حتمية إجبارياً.
4. **قد يبيع بديلاً بدل الرفض** — بدون القاعدة 4 بالبرومبت، طلبُ منتجٍ غير متوفر قد يُقابَل بعرض منتج آخر وكأنه المطلوب.
5. **القدرة العامة محدودة** — متخصص بالبيع/الخدمات؛ خارج هذا النطاق الجودة تتفاوت.
6. **لم يُدرَّب على system prompts** — الاستجابة لها من Gemma الأساسي؛ قد تضعف مع برومبتات معقدة جداً.

## 6. 🔁 سجل الدروس التقنية (لإعادة التدريب مستقبلاً)

- **لا تضع `use_cache=False` أبداً** مع Gemma 4 12B (يفسد مشاركة KV بين الطبقات — transformers issue #45242). كذلك `gradient_checkpointing=False` إلزامي لنفس السبب.
- `target_modules` بقائمة صريحة عبر regex يستثني أبراج الرؤية/الصوت — وليس `"all-linear"`.
- الدمج فوق **bf16 فقط** — الدمج فوق أوزان مكممة (4/8-bit) يراكم أخطاء ويخرب النموذج.
- بعد أي دمج: اختبار مقارنة حتمي قبل/بعد بنفس الأسئلة (Gemma 4 12B حساسة معمارياً).
- لبيانات v5 المقترحة: أمثلة رفض، أمثلة التزام بكتالوج محقون، أمثلة تذكّر تفاصيل الزبون، محادثات أطول (400–800 توكن).



In [ ]:
# -*- coding: utf-8 -*-
"""
النسخة النهائية للنشر - gemma-iraqi-finetune-v2
==================================================
قرارات هذه النسخة (مبنية على 3 جولات اختبار):
1. ❌ حذف الوضع المتوازن نهائياً - انهار مرتين من 3
   (اخترع "كيا رينجر" + gibberish). الحتمي فقط هو المعتمد.
2. ✅ قاعدة جديدة: إذا الزبون گال "اثنين" بدون تحديد الموديل -> اسأله
   (فشل سابق: حسب سعر 2×طن واحد من عنده بدون توضيح)
3. ✅ قاعدة جديدة: ممنوع ادعاء مخزون/ندرة/عروض غير مكتوبة
   (فشل سابق: "عندنا قطعتين بس، ما أريدك تجي بعد وتندم")
4. 🤖 فحص أرقام آلي: كل رقم بالرد لازم يكون موجود حرفياً بالكتالوج
   - هذا هو الـ regression test اللي راح ينلزم مع الـ RAG لاحقاً
"""

import re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# ---------- 1) تحميل النموذج المدمج ----------
MODEL_ID = "ameer4wisam/gemma-iraqi-finetune-v2"   # أو "./gemma-iraqi-merged" محلياً

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",   # مطلوب لمعمارية Gemma 4 12B
)
model.eval()

# ---------- 2) اكتشاف توكن التوقف تلقائياً ----------
_probe_encoded = tokenizer.apply_chat_template(
    [{"role": "user", "content": "هلو"},
     {"role": "assistant", "content": "هلا بيك"}],
    add_generation_prompt=False,
    return_tensors="pt",
    return_dict=True,
)
_probe_ids = _probe_encoded["input_ids"].tolist()[0]

special_ids = set(tokenizer.all_special_ids)
stop_ids = [t for t in _probe_ids[-3:]
            if t in special_ids or "turn" in tokenizer.decode([t])]
if tokenizer.eos_token_id is not None:
    stop_ids.append(tokenizer.eos_token_id)
stop_ids = list(set(stop_ids))
PAD_ID = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else stop_ids[0]
print(f"✅ توكنات التوقف: {stop_ids} -> {[repr(tokenizer.decode([t])) for t in stop_ids]}")

# ---------- 3) إعدادات التوليد: حتمي فقط ----------
# ❌ GEN_BALANCED انحذف نهائياً بعد انهيارين من 3 جولات
#    (كلمات مخترعة، منتجات وهمية "كيا رينجر"، gibberish عشوائي)
# ⚠️ ممنوع: أي do_sample=True | repetition_penalty | temperature
GEN_CONFIG = dict(max_new_tokens=64, do_sample=False)

# ---------- 4) الكتالوج + System Prompt (النسخة النهائية) ----------
CATALOG = """المنتجات المتوفرة حالياً (التزم بيها حرفياً):

الماركات المتوفرة: LG فقط (إذا سأل الزبون عن الماركات، گله عدنا LG بس هسه)

- مكيف LG سبلت طن واحد: 700,000 دينار، ضمان سنتين، استهلاك ~1.0 كيلوواط/ساعة
- مكيف LG سبلت طن ونص: 950,000 دينار، ضمان سنتين، استهلاك ~1.5 كيلوواط/ساعة
  (الفرق بالكهرباء: الطن ونص يصرف حوالي 50% أكثر من الطن الواحد)

- التركيب: 50,000 دينار للجهاز الواحد
- التوصيل داخل بغداد: مجاني | خارج بغداد: 20,000 دينار

خصم شراء قطعتين 5% (المجاميع محسوبة مسبقاً - لا تحسب غيرها):
- 2 × طن واحد: 1,400,000 قبل الخصم -> 1,330,000 بعد الخصم
- 2 × طن ونص: 1,900,000 قبل الخصم -> 1,805,000 بعد الخصم"""

IRAQI_SALES_SYSTEM = f"""أنت بائع عراقي محترف بمحل أجهزة كهربائية.

{CATALOG}

قواعد صارمة:
1. جاوب باللهجة العراقية الأصيلة فقط: شنو، شلون، هسه، اكو، ماكو، زين، خوش، شكد.
2. جاوب قصير ومباشر، جملة أو جملتين مثل البائع الحقيقي.
3. الأسعار والأرقام من القائمة أعلاه فقط - لا تخترع أي رقم ولا تحسب مجاميع جديدة.
4. جاوب على السؤال المطلوب بالضبط:
   - إذا سأل عن الماركات -> عدّد الماركات، لا تذكر السعر.
   - إذا سأل عن استهلاك الكهرباء أو قارن بين موديلين -> جاوب من معلومات الاستهلاك.
   - لا تكرر "ضمان سنتين" إلا إذا انسأل عن الضمان.
5. إذا الزبون گال يريد "اثنين" أو "قطعتين" بدون ما يحدد الموديل،
   اسأله أول: "اثنين طن واحد لو طن ونص؟" - لا تختار عنه.
6. لا تدّعي أي معلومة مخزون أو ندرة أو عروض غير مكتوبة بالقائمة
   (ممنوع: "باقي قطعتين بس"، "العرض ينتهي باچر"، وأمثالها).
7. بِع فقط المنتجات المذكورة بالقائمة. إذا طلب الزبون منتج غير موجود
   (مثلاً ثلاجة والقائمة مكيفات)، گله بصراحة: "والله هذا ماكو عدنا هسه"
   واعرض البديل الموجود إذا مناسب.
8. تذكر تفاصيل الزبون (اسمه، شنو يريد) واستعملها بردودك.
9. إذا ما تعرف معلومة، گول "أتأكدلك" ولا تخترع جواب."""

# ---------- 5) فحص الأرقام الآلي (regression guard) ----------
# القاعدة: أي رقم مالي بالرد لازم يكون موجود حرفياً بالكتالوج.
# نفس الفحص راح يشتغل مع الـ RAG: كل كتالوج محقون = مجموعة أرقام مسموحة.
_ALLOWED_NUMBERS = set(re.findall(r"\d[\d,\.]*", CATALOG))

def check_numbers(reply):
    """يرجع قائمة الأرقام المخترعة (غير الموجودة بالكتالوج).
    نتجاهل الأرقام الصغيرة (<10) لأنها غالباً أعداد قطع أو سنين ضمان."""
    found = re.findall(r"\d[\d,\.]*", reply)
    suspicious = []
    for num in found:
        clean = num.rstrip(".,")
        if clean in _ALLOWED_NUMBERS:
            continue
        # رقم صغير بلا فواصل = عدد قطع/سنين، مو سعر
        if "," not in clean and "." not in clean and len(clean) <= 2:
            continue
        suspicious.append(clean)
    return suspicious

# ---------- 6) دوال الاستدلال ----------
def chat(messages, show_prompt=False):
    """توليد رد واحد (حتمي دائماً). messages بصيغة [{"role": ..., "content": ...}]"""
    enc = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True,
        return_tensors="pt", return_dict=True,
    )
    input_ids = enc["input_ids"].to(model.device)
    attention_mask = enc["attention_mask"].to(model.device)
    if show_prompt:
        print(tokenizer.decode(input_ids[0], skip_special_tokens=False))
    with torch.no_grad():
        out = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            eos_token_id=stop_ids,
            pad_token_id=PAD_ID,
            **GEN_CONFIG,
        )
    return tokenizer.decode(
        out[0][input_ids.shape[1]:], skip_special_tokens=True
    ).strip()


def run_long_conversation(user_turns, system_prompt=IRAQI_SALES_SYSTEM):
    """محادثة متعددة الأدوار + عداد سياق + فحص أرقام آلي لكل رد"""
    messages = []
    violations = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    for i, user_msg in enumerate(user_turns, 1):
        messages.append({"role": "user", "content": user_msg})
        probe = tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, return_dict=True
        )
        ctx_len = len(probe["input_ids"])
        reply = chat(messages)
        messages.append({"role": "assistant", "content": reply})
        print(f"👤 [{i}] (سياق: {ctx_len} توكن): {user_msg}")
        print(f"🤖 [{i}]: {reply}")
        bad_nums = check_numbers(reply)
        if bad_nums:
            violations.append((i, bad_nums))
            print(f"   🚨 أرقام مخترعة (مو بالكتالوج): {bad_nums}")
        print("-" * 60)
    return messages, violations


# ============================================================
# 7) الاختبارات (كلها حتمية)
# ============================================================
all_violations = []

# اختبار أ: مبيعات - يشمل نقاط الفشل السابقة كلها
print("\n" + "🟢 مبيعات - النسخة النهائية".center(60, "=") + "\n")
_, v = run_long_conversation([
    "هلو، شلونكم؟",
    "عندكم مكيفات سبلت؟",
    "شنو الماركات الموجودة؟",           # كان يجاوب بالسعر - انحل
    "شكد سعر الطن الواحد؟",
    "والطن ونص شكد؟",
    "الفرق بالكهرباء بينهم هواي؟",       # كان يجاوب عن الضمان - انحل
    "شكد يصرف الطن الواحد بالساعة؟",
    "زين، والضمان شكد؟",
    "إذا أخذت اثنين تنزلي بالسعر؟",      # ❗ لازم يسأل: أي موديل؟
    "اثنين طن ونص",                      # جديد: بعد ما يسأل، نحدده
    "والتركيب عليكم لو علي؟",
    "التوصيل لأي منطقة؟",
    "زين شوكت أكدر أستلمهم؟",
    "تمام، اتفقنا. فمان الله",
])
all_violations += v

# اختبار ب: الرفض + الذاكرة
print("\n" + "🔴 اختبار الرفض: طلب ثلاجة (غير متوفرة)".center(60, "=") + "\n")
_, v = run_long_conversation([
    "هلو، اسمي أبو علي وأدور على ثلاجة",
    "زين شنو عندكم بدالها؟",
    "طيب تتذكر شنو اسمي وشنو كنت أدور؟",
])
all_violations += v

# اختبار ج: التحيات (حتمي - ما بقى وضع ثاني)
# ❗ نراقب: تحية "صباح الخير" كانت تطلّع ادعاء ندرة ("قطعتين بس")
print("\n" + "🟡 التحيات - حتمي".center(60, "=") + "\n")
_, v = run_long_conversation([
    "هلو شلونك؟",
    "صباح الخير",                        # ❗ كان يدّعي ندرة مخزون
    "شكراً جزيلاً، الله يوفقكم",
    "الله يحفظك، فمان الله",
])
all_violations += v

# ============================================================
# 8) قرار النشر (آلي + يدوي)
# ============================================================
print("\n" + "=" * 60)
if all_violations:
    print(f"🚨 فحص الأرقام: {len(all_violations)} مخالفة - لا تنشر قبل مراجعتها:")
    for turn, nums in all_violations:
        print(f"   الدور [{turn}]: {nums}")
else:
    print("✅ فحص الأرقام: كل الأرقام مطابقة للكتالوج حرفياً")

print("""
📋 مراجعة يدوية قبل النشر:
  [ ] دور "اثنين": سأل "أي موديل؟" ولا حسب من عنده؟
  [ ] تحية "صباح الخير": بلا ادعاء ندرة أو ضغط بيع؟
  [ ] الرفض: گال ماكو ثلاجة + تذكّر أبو علي؟
  [ ] كل التحيات كلام عراقي سليم؟

  ✅ كل النقاط ناجحة + صفر مخالفات أرقام -> انشر (حتمي فقط)
  ⚠️ ملاحظة: أي تعديل مستقبلي بالكتالوج (يدوي أو RAG)
     يغيّر كل المخرجات الحتمية -> أعد هذا السكربت كامل قبل النشر
""")

In [ ]:
from huggingface_hub import hf_hub_download, HfApi
from google.colab import userdata

# نفس اسم السر المستخدم بدفتر التدريب الأصلي (خلية الرفع) — عدّل الاسم إذا مختلف عندك
token = userdata.get('ameerwisam2005')

api = HfApi(token=token)

base_repo = "google/gemma-4-12B-it"
target_repo = "ameer4wisam/gemma-iraqi-finetune-v2"

files_to_copy = ["preprocessor_config.json", "processor_config.json"]

for filename in files_to_copy:
    try:
        local_path = hf_hub_download(repo_id=base_repo, filename=filename, token=token)
        print(f"تحميل {filename} من {base_repo}: نجح")
        api.upload_file(
            path_or_fileobj=local_path,
            path_in_repo=filename,
            repo_id=target_repo,
            repo_type="model",
        )
        print(f"رفع {filename} إلى {target_repo}: نجح")
    except Exception as e:
        print(f"خطأ مع {filename}: {e}")

print("انتهى.")


In [ ]:
import torch
import gc
import os
import shutil
import glob
from safetensors import safe_open
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from huggingface_hub import HfApi, hf_hub_download
from google.colab import userdata

base_model_id = "google/gemma-4-12B-it"
adapter_id = "ameer4wisam/gemma-iraqi-finetune"
target_repo = "ameer4wisam/gemma-iraqi-finetune-v2"
save_dir = "./fixed_gemma_merged"

# 1. تحميل النموذج الأساسي وتخزين معلوماته للتحقق اللاحق
print("1. تحميل النموذج الأساسي لمعرفة المفاتيح المتوقعة...")
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(base_model_id)

base_keys = set(base_model.state_dict().keys())
base_params_count = sum(p.numel() for p in base_model.parameters())
print(f"عدد المفاتيح في النموذج الأساسي: {len(base_keys)}")

# 2. تحميل محول LoRA ودمجه
print("\n2. تحميل محول LoRA ودمجه...")
peft_model = PeftModel.from_pretrained(base_model, adapter_id)
merged_model = peft_model.merge_and_unload()

merged_keys = set(merged_model.state_dict().keys())
print(f"عدد المفاتيح بعد الدمج: {len(merged_keys)}")

# 3. التحقق المبدئي من مفاتيح k_norm
print("\n3. التحقق المبدئي من مفاتيح k_norm...")
k_norm_keys = [k for k in merged_keys if 'k_norm' in k]
print(f"تم العثور على {len(k_norm_keys)} مفتاح يحتوي على 'k_norm'.")
if len(k_norm_keys) == 0:
    print("❌ تحذير: مفاتيح k_norm غير موجودة في الذاكرة بعد الدمج!")

# 4. حفظ النموذج المدموج بشكل آمن
print("\n4. حفظ النموذج المدموج (safe_serialization=True)...")
if os.path.exists(save_dir):
    shutil.rmtree(save_dir)
merged_model.save_pretrained(save_dir, safe_serialization=True)
tokenizer.save_pretrained(save_dir)

# تحرير الذاكرة لتجنب OOM عند إعادة التحميل أو فحص الملفات
del base_model
del peft_model
del merged_model
gc.collect()
torch.cuda.empty_cache()

# 5. التحقق المباشر من ملفات safetensors على القرص
print("\n5. التحقق من سلامة الحفظ عبر فحص ملفات safetensors على القرص مباشرة...")
saved_files = glob.glob(os.path.join(save_dir, "*.safetensors"))
keys_on_disk = set()
for f in saved_files:
    with safe_open(f, framework="pt") as sf:
        keys_on_disk.update(sf.keys())

print(f"عدد المفاتيح المتوقعة: {len(base_keys)}")
print(f"عدد المفاتيح الموجودة فعلياً على القرص: {len(keys_on_disk)}")

# 6. مقارنة النتائج والبحث عن مفاتيح مفقودة أو مشبوهة
print("\n6. مقارنة النتائج...")
success = True
missing_on_disk = base_keys - keys_on_disk

if missing_on_disk:
    print(f"❌ خطأ حقيقي: مفاتيح ناقصة فعلياً من ملف safetensors على القرص ({len(missing_on_disk)} مفتاح):\n{sorted(missing_on_disk)}")
    success = False
else:
    print("✅ كل المفاتيح المتوقعة موجودة فعلياً بملفات safetensors على القرص.")

# تحقق إضافي للقيم المشبوهة (أصفار أو NaN)
suspicious_values = []
for f in saved_files:
    with safe_open(f, framework="pt") as sf:
        for key in sf.keys():
            if "k_norm" in key or "norm" in key:
                tensor = sf.get_tensor(key)
                if torch.isnan(tensor).any() or torch.count_nonzero(tensor) == 0:
                    suspicious_values.append(key)

if suspicious_values:
    print(f"⚠️ مفاتيح موجودة لكن قيمتها مشبوهة (كلها صفر أو NaN): {suspicious_values}")
    success = False
else:
    print("✅ لا توجد قيم مشبوهة (صفر أو NaN) في المفاتيح المفحوصة.")

# 7. الرفع في حال النجاح
if success:
    print("\n✅ الدمج مكتمل ومتحقق منه 100% على مستوى القرص، جاهز للرفع.")

    print("\n7. جلب ملفات preprocessor_config.json و processor_config.json...")
    files_to_copy = ["preprocessor_config.json", "processor_config.json"]
    for filename in files_to_copy:
        try:
            local_path = hf_hub_download(repo_id=base_model_id, filename=filename)
            shutil.copy(local_path, os.path.join(save_dir, filename))
            print(f"✅ تم نسخ {filename} بنجاح.")
        except Exception as e:
            print(f"⚠️ لم يتم العثور على {filename} أو حدث خطأ (قد لا يكون مطلوباً للموديل): {e}")

    print(f"\n8. جاري الرفع إلى {target_repo}...")
    token = userdata.get('ameerwisam2005')
    api = HfApi(token=token)
    api.upload_folder(
        folder_path=save_dir,
        repo_id=target_repo,
        repo_type="model",
        commit_message="Upload verified merged model with exact safetensors disk validation"
    )
    print("🎉 تم الرفع بنجاح!")
else:
    print("\n❌ فشل التحقق. يرجى مراجعة الأخطاء أعلاه. لم يتم رفع النموذج لتجنب استبداله بنسخة معطوبة.")